Lab 8 — KNN: Adult Income Classification

Names: Joonsuh Kim, William Roberts

Dataset: Adult Income
Source: Kaggle — wenruliu/adult-income-dataset

Algorithm: K-Nearest Neighbors (KNN)

Question:
Can we predict whether a person's annual income is above $50,000
using their demographic and employment information?

Input features include age, education, occupation, work hours,
and other information in the dataset.

Target:
Income — $ 50,000 or less , or more than $ 50,000.

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("wenruliu/adult-income-dataset")

print("Path to dataset files:", path)

100%|██████████| 652k/652k [00:00<00:00, 40.6MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/wenruliu/adult-income-dataset/versions/2


In [2]:
from sklearn import datasets
import pandas as pd
import numpy as np


import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.preprocessing import LabelEncoder #encode categoric data
from sklearn.preprocessing import Normalizer #normalize numeric data
from sklearn.model_selection import train_test_split #to split data
from sklearn.neighbors import KNeighborsClassifier #our lovely classifier


In [3]:
data = pd.read_csv(path + "/adult.csv")

data.head()

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K


In [4]:
def object_cols(df):
    return list(df.select_dtypes(include='object').columns)

def numerical_cols(df):
    return list(df.select_dtypes(exclude='object').columns)

In [5]:
obj_col = object_cols(data)
num_col = numerical_cols(data)

In [6]:
le = LabelEncoder()
norm = Normalizer()

In [7]:
for col in obj_col:
    data[col] = le.fit_transform(data[col])

In [8]:
data[num_col] = norm.fit_transform(data[num_col])

In [9]:
data.head()

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,0.000110,4,1.000000,1,0.000031,4,7,3,2,1,0.000000,0.0,0.000176,39,0
1,0.000423,4,1.000000,11,0.000100,2,5,0,4,1,0.000000,0.0,0.000557,39,0
2,0.000083,2,1.000000,7,0.000036,2,11,0,4,1,0.000000,0.0,0.000119,39,1
3,0.000274,4,0.998852,15,0.000062,2,7,0,2,1,0.047898,0.0,0.000249,39,1
4,0.000174,0,1.000000,15,0.000097,4,0,3,4,0,0.000000,0.0,0.000290,39,0


In [10]:
X = data.drop(['income'], axis = 1)
y = data['income']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42)

In [11]:
knn = KNeighborsClassifier() #create a k-NN instance
fit_knn = knn.fit(X_train, y_train) #fitting model by using "fit" method
s_value = X_test[:1] #specific value to predict
print('Predicted class value is : {0}'.format(fit_knn.predict(s_value)))

Predicted class value is : [0]


In [12]:
distance_matrix = fit_knn.kneighbors_graph(s_value, mode = "distance")

ind = distance_matrix.nonzero() # indexes of non-zero elements
ind

(array([0, 0, 0, 0, 0], dtype=int32),
 array([ 2613, 16346,  1138,  9200, 27616], dtype=int32))

In [13]:
k_nearest_classes = [y_train.iloc[val[1]] for val in zip(*distance_matrix.nonzero())]
k_nearest_classes

[np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0)]

In [14]:
print(fit_knn.predict_proba(s_value))

[[1. 0.]]


In [15]:
s_value

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country
7762,0.001691,4,0.999998,11,0.000272,0,8,4,4,0,0.0,0.0,0.001208,39


In [16]:
dis = distance_matrix[distance_matrix.nonzero()] # non-zero distances
dis

k_near = pd.DataFrame()
for val in zip(*distance_matrix.nonzero()):
    k_near = pd.concat([k_near, X_train.iloc[[val[1]]]], ignore_index=True)
k_near

fit_knn.score(X_test, y_test)

0.8200148901848865

Summary

We used KNN to predict whether a person's annual income was above
$50,000 using demographic and employment information. The model
achieved about 82% accuracy on the test data, correctly classifying
roughly 82 out of every 100 people. However, it still made mistakes,
and the preprocessing needs further review before practical use.